好的，我们立刻切换到"2018-2020 年的研究者"视角，那时候大家手里已经有了 Transformer，但开始对前馈网络（FFN）这个"不起眼的组件"动刀。

起初 FFN 只是一个简单的单隐藏层 MLP：`ReLU(xW₁ + b₁)W₂ + b₂`。大家都盯着注意力机制，几乎没人怀疑 FFN 还能怎么改进。但很快，两个发现改变了这一切：
- ReLU 在零点的硬截断导致神经元"死亡"，训练中梯度消失。
- 单一的线性门控路径缺乏对信息的筛选能力——FFN 能不能自己学会"哪些信息该通过"？

这引发了从激活函数到整体结构的一场静悄悄的革命。我们按照 **动机 → 问题 → 公式 → 代码** 的路径，一步步推演。

---

## 1. 原始 FFN：ReLU 的两层 MLP

**Idea**  
Transformer 论文（Vaswani et al., 2017）在每个位置的表示上独立应用一个两层全连接网络，中间用 ReLU 激活。目的：给注意力输出增加非线性，扩展隐藏维度以增加容量。

**Mathematical expression**  
$$
\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2
$$
其中 $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$，$W_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$，一般 $d_{\text{ff}} = 4 \times d_{\text{model}}$。

**Code & Output**  
我们用基础 PyTorch 实现，观察激活值分布。

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
d_model, d_ff = 64, 256
x = torch.randn(4, d_model)  # 模拟4个 token

W1 = torch.randn(d_model, d_ff, requires_grad=True)
b1 = torch.randn(d_ff, requires_grad=True)
W2 = torch.randn(d_ff, d_model, requires_grad=True)
b2 = torch.randn(d_model, requires_grad=True)

# 原始 FFN
hidden = x @ W1 + b1          # [4, 256]
activated = F.relu(hidden)    # ReLU 激活
output = activated @ W2 + b2  # [4, 64]

print("ReLU 后激活值零的比例:", (activated == 0).float().mean().item())
# 通常约有50%的神经元被置零

ReLU 后激活值零的比例: 0.4921875


**痛点**  
ReLU 的硬零输出导致：  
- 神经元一旦进入负区间，梯度为 0，再也无法恢复（死亡 ReLU）。  
- 没有平滑过渡，优化不稳定。

---

## 2. 改用 GELU：平滑的"概率式"门控

**Idea**  
2018 年，Hendrycks & Gimpel 提出 GELU（Gaussian Error Linear Unit），用输入值的高斯累积分布作为"软门控"，既保留了非线性，又给负值一个很小的梯度。直觉：一个神经元的输出，可以看作是输入乘以一个由自身大小决定的"门"——输入越大，门越接近 1；输入很小（负很多），门接近 0。这恰恰是"门控"思想的萌芽。

**Mathematical expression**  
$$
\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5 x \left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715 x^3)\right]\right)
$$
其中 $\Phi$ 是标准正态分布的累积分布函数。实际常使用 tanh 近似。

**Code & Output**  
我们用基础函数实现 GELU 并与 ReLU 对比：

In [ ]:
def gelu_approx(x):
    # 近似公式
    return 0.5 * x * (1.0 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x**3)))

x_vals = torch.linspace(-3, 3, 100)
relu_vals = F.relu(x_vals)
gelu_vals = gelu_approx(x_vals)

# 比较输出
print("负输入区域 ReLU vs GELU:")
print("ReLU(-1)=", F.relu(torch.tensor(-1.0)).item())
print("GELU(-1)=", gelu_approx(torch.tensor(-1.0)).item())
# GELU(-1) 约为 -0.1588，有小幅负激活，且梯度非零

**进步**  
GELU 让所有负值保留了一点梯度，避免神经元永久死亡，训练更平滑。但它的门控机制是**非学习的**：门的强度完全由输入自身决定，模型无法适应性地选择放行或拦截信息。  
这直接引出下一个问题：**能不能让模型显式地学会"门控"？**

---

## 3. GLU（门控线性单元）：引入可学习的门

**Idea**  
2016 年 Dauphin 等人在语言建模中提出 GLU，后来被 Transformer 采纳（如后来的 PaLM）。把一条线性变换路径的输出和另一条路径的输出做逐元素乘积，其中一条路径用 sigmoid 激活作为"门"，另一条路径用线性（或其它激活）作为"值"。这样网络可以学会**根据输入动态地屏蔽或放大某些维度**。

**Mathematical expression**  
GLU 的两个投影为 $U = x W_u + b_u$，$G = x W_g + b_g$，输出为：
$$
\text{GLU}(x) = (x W_g + b_g) \odot \sigma(x W_u + b_u)
$$
通常 $W_g, U \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$，然后通过一个线性投影回到 $d_{\text{model}}$。这个结构取代了原来的两层 MLP，相当于把"激活"换成了带门控的乘法。

也可以把 GLU 写成一个完整的 FFN 块：
$$
\text{FFN}_{\text{GLU}}(x) = \left[ (x W_1) \odot \sigma(x W_2) \right] W_3
$$
这里略去偏置以简化，注意输入同时产生门和值，两者维度相同（$d_{\text{ff}}$）。

**Code & Output**  
我们自己实现 GLU 风格的 FFN，并观察门控的激活模式。

In [ ]:
torch.manual_seed(42)
d_model, d_ff = 64, 256

x = torch.randn(4, d_model)

# GLU 中的三个权重矩阵
W1 = torch.randn(d_model, d_ff, requires_grad=True)   # 值分支
W2 = torch.randn(d_model, d_ff, requires_grad=True)   # 门分支
W3 = torch.randn(d_ff, d_model, requires_grad=True)   # 最终投影

# 计算 GLU
value = x @ W1          # [4, 256]
gate = x @ W2           # [4, 256]
gated_output = value * torch.sigmoid(gate)   # 门控乘法
output = gated_output @ W3   # [4, 64]

print("门激活均值:", torch.sigmoid(gate).mean().item())  # 通常约0.5
print("门激活标准差:", torch.sigmoid(gate).std().item())

**核心改变**  
- 原本 FFN 是 `ReLU(value) W3`，现在变为 `(value * sigmoid(gate)) W3`。  
- 门的可学习性允许模型根据上下文动态抑制或增强某些特征维度，信息流控制更灵活。  
- 但 sigmoid 作为门函数，输出范围 (0,1)，对于深度网络，容易出现梯度饱和（门接近 0 或 1 时梯度很小）。

于是研究者继续追问：能不能找到一个更好的门控激活函数，既平滑又不易饱和？

---

## 4. SwiGLU：GELU 与 GLU 的结合

**Idea**  
2020 年，Shazeer 在《GLU Variants Improve Transformer》中系统研究了多种门控激活，发现把 GLU 的门函数从 sigmoid 换成 **Swish（SiLU）**，即 $x \cdot \sigma(x)$，效果显著提升。Swish 拥有类似 GELU 的平滑非单调性，但计算更简单。这个变体称为 **SwiGLU**，此后被 Llama、PaLM 等大模型标配。

**Mathematical expression**  
SwiGLU 的 FFN 为：
$$
\text{FFN}_{\text{SwiGLU}}(x) = \left[ (x W_1) \odot \text{SiLU}(x W_2) \right] W_3
$$
其中 $\text{SiLU}(z) = z \cdot \sigma(z)$。

注意：此时值分支（$x W_1$）不再经过激活函数，门分支用 SiLU 激活。另外，为了保持参数量与标准 Transformer 一致，通常会缩小隐藏维度（如原来 $d_{\text{ff}}=4d$，现在用 $d_{\text{ff}}=\frac{8}{3}d$ 左右来对齐计算量）。

**Code & Output**  
我们手动实现 SwiGLU，并对比它和 GLU 的门激活分布。

In [ ]:
def silu(x):
    return x * torch.sigmoid(x)

torch.manual_seed(42)
d_model, d_ff = 64, 170  # 8/3 倍左右，保持参数总量接近

x = torch.randn(4, d_model)

W1 = torch.randn(d_model, d_ff, requires_grad=True)  # 值
W2 = torch.randn(d_model, d_ff, requires_grad=True)  # 门（将用 SiLU）
W3 = torch.randn(d_ff, d_model, requires_grad=True)

value = x @ W1
gate_pre = x @ W2
gate_activated = silu(gate_pre)      # SiLU 激活
gated_value = value * gate_activated
output = gated_value @ W3

# 观察门激活的分布
print("SiLU 门均值:", gate_activated.mean().item())
print("SiLU 门标准差:", gate_activated.std().item())
print("SiLU 门最小值:", gate_activated.min().item())
print("SiLU 门最大值:", gate_activated.max().item())
# SiLU 可能出现负值（当 gate_pre 为负且绝对值较大时，约 -0.278），
# 这提供了轻微的正则化效果，并且梯度始终平滑。

**为什么 SwiGLU 更好？**  
1. **梯度流动性**：SiLU 处处可导且梯度非零，避免了 sigmoid 在饱和区域的梯度消失。  
2. **非单调性**：SiLU 在负区域有微小的负值，给予网络更强的表达能力和正则化。  
3. **门控与值的配对**：值分支保留线性，门分支做非线性过滤，信息路径更干净。  
4. 大量实验验证，在相同训练成本下，SwiGLU 能稳定提升 perplexity 和下游任务表现。

---

## 结尾：我们在思路上的演进

回顾这条线索：
- **ReLU**（硬截断）→ 神经元死亡，训练不稳。  
- **GELU**（概率软门控）→ 但门是静态的，只依赖输入本身。  
- **GLU**（引入可学习门控）→ 模型能动态控制信息流，但 sigmoid 门梯度易饱和。  
- **SwiGLU**（SiLU 门控）→ 拥有平滑梯度、非单调性，几乎成为现代 LLM 的标配。

每一步都对应着一个具体的痛点，然后通过"如果……那么……可以怎么改"的思维实验找到数学形式，最终写成代码去实验。现在你手里已经有了 SwiGLU 的最基础实现，可以无缝嵌入我们之前手写的 Transformer 块，替换原来的两层 ReLU MLP，感受它的实际增益。